# Clase 5 — Programación orientada a objetos: creá tus propios tipos de datos

**Curso:** Python sin miedo — UNSL, FCFMyN · **Módulo 3, semana 5**

## Objetivos de hoy

1. Entender qué es una **clase** y un **objeto**: datos y comportamiento viajando juntos.
2. Definir clases con **atributos** y **métodos**: `__init__`, `self`, `__repr__`.
3. Validar datos al construir objetos (**encapsulamiento**).
4. Reutilizar y especializar clases con **herencia** y **polimorfismo**.

Hasta ahora usamos los tipos que Python trae de fábrica: números, strings, listas, diccionarios. Hoy aprendemos a crear los nuestros. Una **clase** es un molde para fabricar objetos que empaquetan *datos* (una muestra tiene código, sitio y pH) junto con *lo que se puede hacer con ellos* (preguntarle si es ácida). Es la forma natural de modelar las entidades de tu investigación: partículas, poblaciones, muestras, grafos.

## 1. ¿Por qué clases? El problema del diccionario viajero

Con lo que sabemos del módulo 2, una muestra de laboratorio sería un diccionario, y sus operaciones, funciones sueltas:

In [ ]:
muestra = {'codigo': 'S-01', 'sitio': 'Norte', 'ph': 6.8}

def es_acida(muestra):
    return muestra['ph'] < 7

print(es_acida(muestra))

Funciona, pero a medida que el proyecto crece aparecen los problemas: el diccionario y sus funciones viven separados, nada impide crear una muestra sin `'ph'` (o con pH 53), y hay que acordarse de qué funciones aplican a qué diccionarios. Una **clase** junta todo en un solo lugar:

```python
class Muestra:                          # el molde (por convención, con Mayúscula)
    def __init__(self, codigo, ...):    # el constructor: cómo se fabrica
        self.codigo = codigo            # los atributos: los datos
    def es_acida(self):                 # los métodos: el comportamiento
        ...
```

In [ ]:
class Muestra:
    """Una muestra de laboratorio con su pH."""

    def __init__(self, codigo, sitio, ph):
        self.codigo = codigo
        self.sitio = sitio
        self.ph = ph

    def es_acida(self):
        return self.ph < 7

    def describir(self):
        tipo = 'ácida' if self.es_acida() else 'no ácida'
        return f'{self.codigo} ({self.sitio}): pH {self.ph}, {tipo}'


# Fabricamos objetos (instancias) llamando a la clase como si fuera una función:
m1 = Muestra('S-01', 'Norte', 6.8)
m2 = Muestra('S-02', 'Sur', 7.4)

print(m1.ph)             # acceder a un atributo: objeto.atributo
print(m1.es_acida())     # llamar a un método: objeto.metodo()
print(m2.describir())

Anatomía de lo que acabamos de escribir:

| Pieza | Qué es |
|---|---|
| `class Muestra:` | define el molde; por convención los nombres de clase van en `Mayúscula` |
| `__init__` | el **constructor**: se ejecuta automáticamente al crear el objeto |
| `self` | el objeto mismo; **siempre** es el primer parámetro de cada método |
| `self.ph = ph` | crea un **atributo**: un dato que vive dentro del objeto |
| `m1.es_acida()` | llama un **método**: una función que viene con el objeto |

Lo de `self` confunde al principio: cuando escribís `m1.es_acida()`, Python lo traduce a `Muestra.es_acida(m1)`. Por eso todo método declara `self` pero al llamarlo no se lo pasás: va solo.

Cada objeto tiene **sus propios** atributos: `m1.ph` y `m2.ph` son independientes. Y como cualquier valor, los objetos pueden ir en listas, diccionarios, pasarse a funciones...

In [ ]:
muestras = [
    Muestra('S-01', 'Norte', 6.8),
    Muestra('S-02', 'Sur', 7.4),
    Muestra('S-03', 'Norte', 5.9),
    Muestra('S-04', 'Sur', 7.1),
]

# Las comprensiones del módulo 2 funcionan igual con objetos:
acidas = [m for m in muestras if m.es_acida()]
print(f'{len(acidas)} muestras ácidas de {len(muestras)}')

ph_norte = [m.ph for m in muestras if m.sitio == 'Norte']
print(f'pH medio en Norte: {sum(ph_norte) / len(ph_norte):.2f}')

## 2. `__repr__`: que tus objetos se presenten solos

Si hacés `print(m1)`, Python muestra algo críptico como `<__main__.Muestra object at 0x...>`. El método especial `__repr__` define cómo se muestra el objeto — y Python lo usa automáticamente en `print`, en listas, en la consola:

In [ ]:
class Muestra:
    """Una muestra de laboratorio con su pH."""

    def __init__(self, codigo, sitio, ph):
        self.codigo = codigo
        self.sitio = sitio
        self.ph = ph

    def __repr__(self):
        return f'Muestra({self.codigo!r}, {self.sitio!r}, ph={self.ph})'

    def es_acida(self):
        return self.ph < 7


m1 = Muestra('S-01', 'Norte', 6.8)
print(m1)                      # usa __repr__
print([m1, Muestra('S-02', 'Sur', 7.4)])   # también dentro de listas

Los métodos que empiezan y terminan con doble guion bajo (`__init__`, `__repr__`, y hay muchos más: `__eq__`, `__len__`, `__add__`...) se llaman **métodos especiales** o *dunder*: son los ganchos con los que tus clases se enchufan al resto del lenguaje. Convención útil para `__repr__`: que se parezca al código que recrearía el objeto.

### 🖊️ Tu turno 1

Escribí una clase `Circulo` con atributo `radio` y métodos `area()`, `perimetro()` y `__repr__`. Creá un círculo de radio 2 y mostrá sus tres resultados (pista: `import math` y usá `math.pi`).

Verificación: con radio 2, área ≈ 12.566 y perímetro ≈ 12.566 (sí, dan casi lo mismo: coincidencia exclusiva del radio 2 — ¿ves por qué?).

In [ ]:
# Tu código acá


## 3. Validar al construir: encapsulamiento

Una de las grandes ventajas de las clases: el constructor es el **punto de control** de tus datos. Si una muestra con pH 53 no tiene sentido, que sea imposible crearla — usamos el `raise` que aprendimos en la clase 4:

In [ ]:
class Muestra:
    """Muestra de laboratorio. El pH debe estar entre 0 y 14."""

    def __init__(self, codigo, sitio, ph):
        if not 0 <= ph <= 14:
            raise ValueError(f'pH fuera de rango: {ph} (debe estar entre 0 y 14)')
        self.codigo = codigo
        self.sitio = sitio
        self.ph = ph

    def __repr__(self):
        return f'Muestra({self.codigo!r}, {self.sitio!r}, ph={self.ph})'

    def es_acida(self):
        return self.ph < 7


print(Muestra('S-01', 'Norte', 6.8))    # esta se crea bien

try:
    Muestra('S-99', 'Norte', 53)        # esta no debe existir
except ValueError as e:
    print(f'Rechazada: {e}')

A esta idea de proteger el estado interno del objeto se la llama **encapsulamiento**. Python lo maneja con convenciones más que con prohibiciones:

- Un atributo que empieza con guion bajo (`self._calibracion`) es un cartel de *"interno: no tocar desde afuera"*. Python no lo impide, pero todo el ecosistema respeta la señal.
- Para quien quiera profundizar: el decorador `@property` permite atributos calculados y validación al asignar. No lo vamos a necesitar en el curso, pero lo vas a cruzar leyendo código ajeno.

## 4. Herencia: especializar sin copiar

Supongamos que modelamos los sensores del laboratorio. Todos hacen lo mismo en esencia (registrar lecturas, calcular el promedio), pero cada tipo tiene sus particularidades. ¿Copiamos y pegamos la clase entera para cada tipo de sensor? No: definimos una clase **base** con lo común, y clases **hijas** que la especializan.

In [ ]:
class Sensor:
    """Sensor genérico que acumula lecturas numéricas."""

    def __init__(self, nombre, unidad):
        self.nombre = nombre
        self.unidad = unidad
        self.lecturas = []

    def __repr__(self):
        return f'{type(self).__name__}({self.nombre!r}, {len(self.lecturas)} lecturas)'

    def registrar(self, valor):
        self.lecturas.append(valor)

    def promedio(self):
        if not self.lecturas:
            raise ValueError(f'{self.nombre}: sin lecturas todavía')
        return sum(self.lecturas) / len(self.lecturas)


s = Sensor('caudal-01', 'L/s')
s.registrar(12.1)
s.registrar(11.8)
print(s, '→ promedio', s.promedio())

In [ ]:
class SensorPH(Sensor):
    """Sensor de pH: unidad fija y lecturas validadas al rango 0-14."""

    def __init__(self, nombre):
        super().__init__(nombre, 'pH')      # reusa el constructor de Sensor

    def registrar(self, valor):             # REEMPLAZA al de Sensor (override)
        if not 0 <= valor <= 14:
            raise ValueError(f'pH inválido: {valor}')
        super().registrar(valor)            # ...pero delega el trabajo común

    def es_acido(self):                     # método NUEVO, solo de los pH
        return self.promedio() < 7


ph = SensorPH('ph-lab-2')
ph.registrar(6.8)
ph.registrar(6.5)
print(ph, '→ promedio', ph.promedio(), '→ ¿ácido?', ph.es_acido())

try:
    ph.registrar(53)
except ValueError as e:
    print(f'Rechazada: {e}')

Lo que acaba de pasar:

- `class SensorPH(Sensor):` — `SensorPH` **hereda** todo lo de `Sensor`: `promedio()` le funciona gratis, sin reescribirlo.
- `super()` es "mi clase madre": `super().__init__(...)` y `super().registrar(...)` reutilizan el código de `Sensor` en lugar de duplicarlo.
- Redefinir un método heredado (como `registrar`) se llama **sobreescribir** (*override*): la hija decide su propia versión.

## 5. Polimorfismo: el mismo código para toda la familia

La consecuencia más poderosa de la herencia: el código que trabaja con `Sensor` funciona con **cualquier** sensor, presente o futuro, y cada objeto responde con *su* versión de cada método. Eso es el **polimorfismo**:

In [ ]:
tablero = [s, ph]    # un Sensor genérico y un SensorPH, mezclados

for sensor in tablero:
    print(f'{sensor.nombre}: promedio {sensor.promedio():.2f} {sensor.unidad}')

# Este bucle no pregunta de qué clase es cada uno: solo pide promedio()
# y cada objeto responde como sabe. Mañana agregás SensorTemperatura
# y este mismo código lo muestra sin tocar una línea.

### 🖊️ Tu turno 2

Escribí `SensorTemperatura(Sensor)`: unidad fija `'°C'`, y un método nuevo `amplitud()` que devuelva `max(lecturas) - min(lecturas)`. Creá uno, registrá las lecturas `18.2, 22.5, 19.8, 24.1` y mostrá su promedio y su amplitud (verificación: promedio 21.15, amplitud 5.9). Agregalo al `tablero` y volvé a correr el bucle polimórfico.

In [ ]:
# Tu código acá


## 6. Práctica integradora (segunda parte de la clase): modelar entidades científicas

### Ejercicio 1 — Partícula

Escribí una clase `Particula` con atributos `masa`, posición `x, y` y velocidad `vx, vy`, y con estos métodos:

- `paso(dt)`: avanza la posición un intervalo `dt` (`x` aumenta `vx * dt`, ídem `y`).
- `energia_cinetica()`: devuelve $\frac{1}{2} m (v_x^2 + v_y^2)$.
- `__repr__`: algo legible, p. ej. `Particula(m=2, pos=(0.30, 0.40))`.

Creá una partícula con masa 2 en el origen con velocidad `(3, 4)`, avanzala 10 pasos de `dt = 0.1` y mostrá posición final y energía. Verificación: posición `(3.0, 4.0)`, energía cinética `25.0`.

In [ ]:
# Ejercicio 1


### Ejercicio 2 — Herencia: partícula cargada

Escribí `ParticulaCargada(Particula)` que agregue el atributo `carga` (usando `super().__init__` para no repetir el resto), un método `es_positiva()`, y un `__repr__` propio que incluya la carga. Creá una con carga `-1.6e-19`, movela y verificá que `paso` y `energia_cinetica` siguen funcionando sin haberlos reescrito.

In [ ]:
# Ejercicio 2


### Ejercicio 3 — Poblaciones y polimorfismo

Modelá dos modelos clásicos de crecimiento poblacional:

- `Poblacion(nombre, tamanio, tasa)` con método `paso()`: crecimiento exponencial, el tamaño se multiplica por `(1 + tasa)`.
- `PoblacionLogistica(Poblacion)` que agrega `capacidad` y sobreescribe `paso()`: el tamaño aumenta `tasa * tamanio * (1 - tamanio / capacidad)` (crecimiento logístico).

Creá una de cada (ambas con tamaño inicial 100 y tasa 0.3; la logística con capacidad 1000) y simulá 20 pasos **con un único bucle polimórfico**, mostrando ambos tamaños cada 5 pasos. Verificación aproximada en el paso 20: exponencial ≈ 19005, logística ≈ 983 (se frena cerca de la capacidad).

In [ ]:
# Ejercicio 3


### Ejercicio 4 (desafío) — Grafo

Un grafo (red) se puede guardar como un diccionario `{vertice: conjunto_de_vecinos}`. Escribí una clase `Grafo` con métodos:

- `agregar_arista(a, b)`: conecta `a` y `b` en ambos sentidos (creando los vértices si no existen — pista: `setdefault` o `if not in`).
- `vecinos(v)` y `grado(v)` (cantidad de vecinos).
- `num_vertices()` y `num_aristas()` (¡ojo: cada arista conecta dos vértices — no la cuentes dos veces!).

Probalo con esta red de coautorías: Ana–Bruno, Ana–Carla, Bruno–Carla, Carla–Diego. Verificación: 4 vértices, 4 aristas, grado de Carla = 3, grado de Diego = 1.

In [ ]:
# Ejercicio 4


## Resumen de la clase

| Concepto | Sintaxis clave |
|---|---|
| Definir una clase | `class Muestra:` |
| Constructor | `def __init__(self, ...):` |
| Atributo / método | `self.ph = ph` / `def es_acida(self):` |
| Cómo se muestra | `def __repr__(self):` |
| Validar al construir | `raise ValueError(...)` dentro de `__init__` |
| Heredar | `class SensorPH(Sensor):` |
| Reusar a la madre | `super().__init__(...)`, `super().metodo(...)` |
| Polimorfismo | el mismo bucle llama a `metodo()` y cada clase responde a su manera |

## Para la próxima semana

- Empezá la [guía de ejercicios del módulo 3](guia-ejercicios-modulo-3.ipynb): ya podés resolver las **Partes A y B** (clases y herencia) — entrega: **viernes 30/10/2026**.
- La próxima clase cerramos el módulo: **módulos y bibliotecas estándar** — cómo sacar estas clases del notebook, guardarlas en archivos `.py` reutilizables y aprovechar la biblioteca que Python trae de fábrica (`math`, `random`, `datetime`, `os`).